# Ordenacao e Busca na Pratica

## Live 3 - Data Structure Strategy and Implementation

Nesta aula vamos ligar algoritmos de ordenacao e busca a um caso do mundo real: um mini mecanismo de busca inspirado em Elasticsearch.

A ideia nao e reproduzir o Elasticsearch completo, e sim entender os fundamentos de indice, consulta e ranking.

In [ ]:
from collections import defaultdict, Counter
import re

print('Ambiente pronto para estudar ordenacao e busca.')

---

## 1. O problema real

Imagine uma plataforma educacional com centenas ou milhares de textos. O usuario digita algo como:

- `python estrutura`
- `busca binaria`
- `arvore bst`

Se fizermos busca linear em todos os textos, a resposta fica cada vez mais cara.

### Nosso conjunto de documentos

Vamos criar uma pequena base textual para simular artigos ou paginas indexadas.

In [ ]:
documentos = {
    1: 'python para estrutura de dados e algoritmos',
    2: 'introducao a busca binaria em listas ordenadas',
    3: 'como funciona uma arvore binaria de busca bst',
    4: 'elasticsearch usa indice invertido para acelerar busca textual',
    5: 'ordenacao com merge sort e quick sort',
    6: 'hash tables e busca rapida por chave',
    7: 'estruturas de dados em python com exemplos reais',
    8: 'busca textual ranking e relevancia em motores de pesquisa',
    9: 'como indexar documentos e consultar termos rapidamente',
}

print('Quantidade de documentos:', len(documentos))
for doc_id, texto in documentos.items():
    print(doc_id, '->', texto)

---

## 2. Busca linear

Primeiro vamos fazer a forma mais direta: procurar termo por termo em todos os documentos.

In [ ]:
def busca_linear(documentos, termo):
    resultados = []
    comparacoes = 0
    for doc_id, texto in documentos.items():
        comparacoes += 1
        if termo.lower() in texto.lower():
            resultados.append(doc_id)
    return resultados, comparacoes

resultado, comparacoes = busca_linear(documentos, 'busca')
print('Documentos encontrados:', resultado)
print('Comparacoes realizadas:', comparacoes)

Isso funciona, mas a cada nova consulta varremos tudo. Em uma base muito grande, essa estrategia nao escala bem.

---

## 3. Tokenizacao

Motores de busca quebram os textos em termos. Vamos fazer uma tokenizacao simples: letras minusculas e separacao por palavras.

In [ ]:
def tokenizar(texto):
    return re.findall(r'[a-zA-ZÀ-ÿ0-9]+', texto.lower())

print(tokenizar(documentos[4]))

---

## 4. Construindo um indice invertido

O indice invertido mapeia cada termo para a lista de documentos em que ele aparece.

Isso e a base conceitual de sistemas como Elasticsearch.

In [ ]:
indice = defaultdict(list)

for doc_id, texto in documentos.items():
    termos_unicos = set(tokenizar(texto))
    for termo in termos_unicos:
        indice[termo].append(doc_id)

for termo in ['python', 'busca', 'estrutura', 'elasticsearch']:
    print(termo, '->', indice[termo])

### O ganho estrutural

Agora nao precisamos olhar todos os documentos para cada busca. Consultamos diretamente o termo no dicionario e obtemos os candidatos.

---

## 5. Consulta com multiplos termos

Vamos construir uma busca simples que:

1. tokeniza a consulta
2. recupera documentos por termo
3. soma ocorrencias para ranquear

In [ ]:
def buscar_indice(consulta, documentos, indice):
    termos = tokenizar(consulta)
    score = Counter()

    for termo in termos:
        for doc_id in indice.get(termo, []):
            score[doc_id] += documentos[doc_id].lower().count(termo)

    return score.most_common()

resultado = buscar_indice('python estrutura busca', documentos, indice)
print(resultado)

### Interpretacao

O ranking aqui e muito simples: quanto mais vezes os termos aparecem, maior o score.

Motores reais fazem muito mais do que isso, mas a estrutura principal ja aparece aqui.

In [ ]:
consulta = 'python estrutura busca'
resultado = buscar_indice(consulta, documentos, indice)

print('Consulta:', consulta)
print()
for doc_id, score in resultado:
    print(f'doc {doc_id} | score={score} | {documentos[doc_id]}')

---

## 6. Onde entra ordenacao?

Buscar e so metade do problema. Depois precisamos ordenar os resultados.

No nosso exemplo, ordenamos por score de relevancia. Em outros sistemas, podemos ordenar por:

- data
- preco
- popularidade
- nota
- distancia

In [ ]:
produtos = [
    {'nome': 'Notebook A', 'preco': 4200, 'nota': 4.5},
    {'nome': 'Notebook B', 'preco': 3500, 'nota': 4.8},
    {'nome': 'Notebook C', 'preco': 5100, 'nota': 4.2},
]

ordenados_por_preco = sorted(produtos, key=lambda x: x['preco'])
ordenados_por_nota = sorted(produtos, key=lambda x: x['nota'], reverse=True)

print('Por preco:')
for p in ordenados_por_preco:
    print(p)

print('\nPor nota:')
for p in ordenados_por_nota:
    print(p)

---

## 7. Ligando com Elasticsearch

Nosso mini projeto nao implementa tudo o que um search engine faz, mas ja ilustra fundamentos reais:

- tokenizacao
- indice invertido
- recuperacao por termo
- ranking
- ordenacao dos resultados

Elasticsearch adiciona distribuicao, analise linguistica, filtros estruturados, shards, relevancia mais sofisticada e muitas outras camadas.

---

## 8. Conclusoes

O que vimos nesta aula:

1. Busca linear e simples, mas cresce mal
2. Ordenacao e indice mudam drasticamente o custo das consultas
3. O indice invertido e uma estrutura central em busca textual
4. Buscar e ordenar resultados sao partes complementares do mesmo problema

Resumo:
- **Ordenar** prepara ou organiza resposta
- **Buscar** depende da estrutura escolhida
- **Indices** tornam sistemas reais muito mais eficientes